# 2026.10.15(목) 22h25~

encoder = self-attention, 출력층에서 2개 출력을 내보냄?
vs
decoder = masked? self-attention, 출력층에서 vocabulary size만큼 예측

cf. 내가 수업 시간에 찾은 것
1. use_causal_mask=True
2. 데이터 준비 (학습을 위해 문장을 한 칸씩 밀어서 정답지 생성)

## 인코더 vs 디코더 코드 차이점 정리 (인코더 노트북 ↔ 이 노트북)

> 위쪽에 직접 적은 메모를 코드와 대조해서 정리했어요. 맞는 부분, 보완할 부분을 표 아래에 따로 표시했습니다.
> 아래 표의 "인코더"는 `FCSS_TransFormer_예제_인코더.ipynb`(로그 위험도 분류), "디코더"는 이 노트북(다음 단어 생성)의 코드 기준이에요.

### 1. 한눈에 보는 비교표

| # | 항목 | 인코더 노트북 | 디코더 노트북 (이 파일) |
|---|---|---|---|
| 1 | **하는 일** | 문장 전체를 읽고 **분류** (정상/위험) | 앞 단어들을 보고 **다음 단어 예측** (문장 생성) |
| 2 | **Self-Attention의 시야** | 모든 위치를 다 봄 (앞·뒤 모두, 양방향) | **자기 위치와 그 앞쪽만** 봄 (`use_causal_mask=True`) |
| 3 | **MultiHeadAttention 호출** | `MultiHeadAttention(...)(inputs, inputs)` | `MultiHeadAttention(...)(inputs, inputs, use_causal_mask=True)` |
| 4 | **블록 구조** (Attention → Add&Norm → FFN → Add&Norm) | 동일 | 동일 (마스크 인자 하나만 다름) |
| 5 | **블록 뒤 출력층** | `GlobalAveragePooling1D` → `Dropout` → `Dense(20, relu)` → `Dense(2, softmax)` | `Dense(vocab_size, softmax)` 하나 (**풀링 없음**) |
| 6 | **출력 모양** | `(배치, 2)` — 문장당 확률 2개 | `(배치, 19, 2000)` — **모든 위치마다** 사전 크기만큼의 확률 |
| 7 | **출력 개수가 정해지는 이유** | 클래스 개수 (정상/위험 = 2) | 단어 사전 크기 (다음 단어 후보 전체 = 2000) |
| 8 | **정답(라벨)** | 사람이 붙인 라벨 `y = [0,0,…,1,1]` (문장당 1개) | 라벨 불필요. 입력 문장을 **한 칸 민 것**이 정답 (`y = text[:, 1:]`) |
| 9 | **입력 X** | `vectorize_layer(문장)` 전체 (길이 `maxlen` = 20) | `text[:, :-1]` — 마지막 단어 제외 (길이 `maxlen-1` = 19) |
| 10 | **정답 Y 모양** | `(20,)` — 문장 20개에 정수 20개 (문장당 1개) | `(400, 19)` — X와 같은 모양의 정수들 |
| 11 | **손실 함수** | `sparse_categorical_crossentropy` (문장 1개당 1번 계산) | `sparse_categorical_crossentropy` (**위치마다** 계산해 평균) |
| 12 | **추론(사용) 방식** | `model.predict()` **한 번**으로 끝 | 단어를 하나 예측 → 입력에 붙임 → 다시 예측하는 **반복 루프** (`generate_text`) |
| 13 | **TokenAndPositionEmbedding** | 같음 (공통 부품). 단, 길이 인자가 `maxlen` | 같음. 길이 인자가 `maxlen-1` (입력이 한 칸 짧아서) |

(숫자: 인코더의 `(배치, 2)`와 디코더의 `(None, 19, 2000)`은 두 노트북 모델을 직접 만들어 `output_shape`로 확인한 값이에요. X는 `(400, 19)`, y도 `(400, 19)`.)

### 2. 내가 적은 메모 점검

| 메모 | 판정 | 보완 |
|---|---|---|
| 인코더 = self-attention | ✅ 맞음 | 정확히는 **마스크 없는** self-attention (양방향) |
| 인코더: 출력층에서 2개 출력 | △ 절반만 | "2개"는 **인코더 구조의 특징이 아니라 이 예제가 정상/위험 2분류라서**입니다. 인코더 블록 자체의 출력은 `(배치, 20, 32)` (단어마다 32차원 벡터)이고, 그걸 `GlobalAveragePooling1D`로 문장 하나로 압축한 뒤 `Dense(2)`를 붙여서 2개가 나온 거예요. 3분류면 3개가 됩니다. |
| 디코더 = masked self-attention | ✅ 맞음 | 코드로는 `use_causal_mask=True` (=내가 찾은 것 1번). 현재 위치보다 뒤의 단어를 못 보게 가림 |
| 디코더: 출력층에서 vocabulary size만큼 예측 | ✅ 맞음 | 매 위치에서 "다음 단어가 사전의 어떤 단어일 확률"을 `Dense(vocab_size, softmax)`로 계산. **풀링 없이 모든 위치**에서 예측한다는 점이 핵심 |
| 내가 찾은 것 1. `use_causal_mask=True` | ✅ | 아래 3절 참고 |
| 내가 찾은 것 2. 문장을 한 칸씩 밀어 정답지 생성 | ✅ | 아래 4절. 이것이 곧 teacher forcing (`010`/`011` 노트북에서 배운 "정답을 오른쪽으로 한 칸 민 디코더 입력"과 같은 아이디어) |

### 3. 마스크가 정말 하는 일 (직접 실험)

같은 입력에서 **마지막 단어(6번째)만 바꿨을 때** 각 위치의 Attention 출력이 얼마나 변하는지 측정한 값이에요 (0이면 영향 없음).

| | 1번째 | 2번째 | 3번째 | 4번째 | 5번째 | 6번째 |
|---|---|---|---|---|---|---|
| 마스크 없음 (인코더 방식) | 3.09 | 0.98 | 2.54 | 1.93 | 12.09 | 5.96 |
| `use_causal_mask=True` (디코더 방식) | **0** | **0** | **0** | **0** | **0** | 1.80 |

- 마스크가 없으면 6번째 단어가 앞쪽 모든 위치의 결과를 바꿔요. 즉 **앞 단어의 표현이 뒷 단어를 훔쳐보는 상태**입니다.
- 마스크가 있으면 뒤 단어를 바꿔도 앞 위치 출력은 **완전히 그대로**예요.
- 이게 중요한 이유: 학습할 때 정답(다음 단어)이 입력에 같이 들어가 있으니, 가리지 않으면 모델이 **답을 베끼기만** 하고 예측을 배우지 않아요. 마스크로 "각 위치는 자기 앞까지만 보고 다음 단어를 맞혀라"는 조건을 만들어 줍니다.
- 이 성질 덕분에 `generate_text`에서 아직 채워지지 않은 뒷부분이 0(빈 칸)이어도 `predictions[0, i, :]`(i번째 위치 예측)는 뒷부분의 영향을 받지 않아요.

### 4. 한 칸 민 데이터가 만드는 학습 (예: "장비 가동 상태 정상 유지")

```
text_ds   = [장비, 가동, 상태, 정상, 유지, 0, 0, ...]      (길이 20)
X_train   = [장비, 가동, 상태, 정상, 유지, 0, ...]         text_ds[:, :-1]  (길이 19)
y_train   = [가동, 상태, 정상, 유지, 0 , 0, ...]            text_ds[:, 1:]   (길이 19)

위치 1: 입력 "장비"                → 정답 "가동"
위치 2: 입력 "장비 가동"           → 정답 "상태"
위치 3: 입력 "장비 가동 상태"      → 정답 "정상"
```

- 문장 하나에서 위치 수만큼의 연습문제가 **한 번에** 나옵니다. (마스크가 있어서 가능)
- 별도 라벨링이 필요 없고 문장 자체가 정답을 포함하는 방식이에요 (자기지도 학습).
- 인코더 분류는 문장 하나당 정답이 하나(0 또는 1)라 사람이 라벨을 붙여야 했어요.

### 5. 추론 방식 차이 (`generate_text`)

```
Step 1: [장비]                 → "가동" 예측 → 입력에 붙임
Step 2: [장비, 가동]           → "상태" 예측 → 붙임
Step 3: [장비, 가동, 상태]     → "정상" 예측 → ...
```

- 학습은 문장 전체를 **한꺼번에**(병렬), 생성은 **한 단어씩**(순차)이에요. 학습과 달리 생성할 때는 정답을 모르니 모델이 방금 낸 예측이 다음 입력이 돼요.
- 실행 결과: "장비" → "장비 가동 상태 정상 유지", "온도" → "온도 수치 매우 안정". 반면 사전에 없는 "날씨"는 아무것도 생성하지 못했어요(학습 데이터에 없는 단어).

### 6. 이 디코더 코드의 한계와 "논문 속 디코더"와의 차이 (중요)

| | 논문(Vaswani 2017)의 디코더 층 | 이 노트북의 디코더 블록 |
|---|---|---|
| 서브층 개수 | **3개**: ① Masked Self-Attention, ② **Encoder-Decoder Attention**, ③ FFN | **2개**: ① Masked Self-Attention, ② FFN |
| 인코더 출력 참조 | ②번에서 Query=디코더, Key/Value=**인코더 출력** | **없음** |
| 입력 | 번역 같은 "출발 문장"(인코더 입력) + "목표 문장"(디코더 입력) | 문장 하나뿐 |

- 이 노트북은 인코더 없이 **디코더만으로** 구성된 모델(GPT 같은 방식)이라서 ② 층이 필요 없어서 빠져 있어요. 논문 §3.1: 디코더는 인코더와 같은 두 서브층에 "인코더 출력에 대한 multi-head attention"을 세 번째 서브층으로 추가. 논문의 번역 모델에서는 이 층이 `011`에서 배운 Seq2Seq의 Attention 역할을 합니다.
- 반대로 인코더 노트북은 디코더 없이 인코더만 쓰는 **분류 모델**입니다(BERT류 방식).
- 정리하면 두 노트북은 논문 구조의 "일부분"을 따로 떼어 쓴 거예요.
  - 인코더만 → 문장 이해/분류
  - 디코더만 → 문장 생성
  - 인코더 + 디코더 (+ 교차 어텐션) → 번역, 요약처럼 "입력 문장을 보고 다른 문장을 생성"

(위 "GPT 같은 방식", "BERT류" 표현은 논문이 아니라 제 배경 지식이고, 논문 구조와의 대응은 `transformer.pdf` §3.1, Figure 1 기준입니다. 이 폴더의 PDF에서는 텍스트만 읽었고 Figure 1 그림은 직접 보지 못했어요.)

### 7. 참고: 두 코드 모두 아직 안 다룬 것
- **패딩 마스크**: 문장 뒤에 채운 0(패딩)도 Attention 대상에 그대로 포함돼요. 실제 서비스에서는 패딩 위치를 가리는 마스크를 따로 넣습니다. (이 예제는 문장이 비슷해서 영향이 작음. 이 항목은 코드를 읽고 한 추정이며 실험으로 확인하지는 않았어요.)
- **샘플 수**: 디코더 학습 데이터는 4개 문장을 100번 반복한 것(400개)이라 새로운 문장을 만들어 내는 게 아니라 **외운 문장을 이어 붙이는** 수준이에요.


In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

# 1. 위치 정보 포함 임베딩 (인코더와 동일한 공통 부품)
class TokenAndPositionEmbedding(layers.Layer):
    def __init__(self, maxlen, vocab_size, embed_dim):
        super().__init__()
        self.token_emb = layers.Embedding(input_dim=vocab_size, output_dim=embed_dim)
        self.pos_emb = layers.Embedding(input_dim=maxlen, output_dim=embed_dim)

    def call(self, x):
        maxlen = tf.shape(x)[-1]
        positions = tf.range(start=0, limit=maxlen, delta=1)
        positions = self.pos_emb(positions)
        x = self.token_emb(x)
        return x + positions

# 2. 디코더 블록 (미래를 보지 못하게 가리는 'Causal Mask'가 핵심)
def transformer_decoder_block(embed_dim, num_heads, ff_dim, rate=0.1):
    inputs = layers.Input(shape=(None, embed_dim))
    
    # [핵심] use_causal_mask=True : 현재 시점보다 뒤에 있는 단어는 가림
    attn_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=embed_dim
    )(inputs, inputs, use_causal_mask=True)
    
    attn_output = layers.Dropout(rate)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(inputs + attn_output)
    
    ffn_output = layers.Dense(ff_dim, activation="relu")(out1)
    ffn_output = layers.Dense(embed_dim)(ffn_output)
    ffn_output = layers.Dropout(rate)(ffn_output)
    
    return tf.keras.Model(inputs=inputs, outputs=layers.LayerNormalization(epsilon=1e-6)(out1 + ffn_output))

# 3. 데이터 준비 (학습을 위해 문장을 한 칸씩 밀어서 정답지 생성)
vocab_size, maxlen, embed_dim = 2000, 20, 32

# 학습용 샘플 문장 (반복 학습을 위해 양을 늘림)
sentences = [
    "장비 가동 상태 정상 유지",
    "현재 온도 수치 매우 안정",
    "모터 진동 발생 점검 필요",
    "긴급 상황 즉시 전원 차단"
] * 100 

vectorize_layer = layers.TextVectorization(max_tokens=vocab_size, output_sequence_length=maxlen)
vectorize_layer.adapt(sentences)

# 입력 데이터(X)와 정답 데이터(Y) 준비
# 디코더 학습의 핵심: "A B C"를 보고 "B C D"를 맞히도록 구성
text_ds = vectorize_layer(sentences)
X_train = text_ds[:, :-1] # 마지막 단어 제외
y_train = text_ds[:, 1:]  # 첫 번째 단어 제외 (한 칸씩 밀린 정답)

# 4. 디코더 모델 조립
inputs = layers.Input(shape=(maxlen-1,))
x = TokenAndPositionEmbedding(maxlen-1, vocab_size, embed_dim)(inputs)
x = transformer_decoder_block(embed_dim, 2, 32)(x)
outputs = layers.Dense(vocab_size, activation="softmax")(x)

decoder_model = tf.keras.Model(inputs=inputs, outputs=outputs)
decoder_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy")

print("--- 디코더 학습 시작 ---")
decoder_model.fit(X_train, y_train, epochs=50, batch_size=16, verbose=0)
print("--- 학습 완료 ---\n")

# 5. 문장 생성 루프 (Inference Loop)
def generate_text(seed_text, max_gen_len=5):
    vocab = vectorize_layer.get_vocabulary()
    # 시작 단어를 숫자로 변환
    input_tokens = np.zeros((1, maxlen-1))
    initial_tokens = vectorize_layer([seed_text]).numpy()[0]
    
    # 첫 단어 채우기
    current_len = 0
    for val in initial_tokens:
        if val != 0:
            input_tokens[0, current_len] = val
            current_len += 1
            
    result = seed_text
    print(f"🌱 시작: {seed_text}")

    for i in range(current_len - 1, current_len - 1 + max_gen_len):
        # 다음에 올 모든 단어의 확률 예측
        predictions = decoder_model.predict(input_tokens, verbose=0)
        
        # 현재 시점(i)에서 가장 확률 높은 단어 선택
        next_token_id = np.argmax(predictions[0, i, :])
        next_word = vocab[next_token_id]
        
        if next_word == "" or next_word == "[UNK]": break
            
        result += " " + next_word
        print(f"   ㄴ Step {i - current_len + 2}: {result}")
        
        # 다음 루프를 위해 예측한 단어를 입력창에 추가
        if i + 1 < maxlen - 1:
            input_tokens[0, i + 1] = next_token_id
            
    return result




--- 디코더 학습 시작 ---
--- 학습 완료 ---

🌱 시작: 장비
   ㄴ Step 1: 장비 가동
   ㄴ Step 2: 장비 가동 상태
   ㄴ Step 3: 장비 가동 상태 정상
   ㄴ Step 4: 장비 가동 상태 정상 유지


'장비 가동 상태 정상 유지'

In [3]:
# 테스트 실행
generate_text("장비")
generate_text("온도")
generate_text("날씨")

🌱 시작: 장비
   ㄴ Step 1: 장비 가동
   ㄴ Step 2: 장비 가동 상태
   ㄴ Step 3: 장비 가동 상태 정상
   ㄴ Step 4: 장비 가동 상태 정상 유지
🌱 시작: 온도
   ㄴ Step 1: 온도 수치
   ㄴ Step 2: 온도 수치 매우
   ㄴ Step 3: 온도 수치 매우 안정
🌱 시작: 날씨


'날씨'